In [0]:
from pyspark.sql.functions import col, row_number, to_timestamp, current_timestamp,lit
from pyspark.sql.window import Window
from datetime import date

In [0]:
dbutils.widgets.text("load_date", "")
load_date = dbutils.widgets.get("load_date")

if load_date:
    load_date=date.fromisoformat(load_date)
else:
    load_date=date.today()

In [0]:
# Deduplicates patient records and maintains current patient versions in the Silver Delta table.
patient_deduplicate = spark.sql('''select active, address,	birthDate, communication, contact, extension, gender, generalPractitioner, id,	identifier,	managingOrganization,maritalStatus,meta,name,resourceType,	telecom, text,file_path,	extraction_timestamp,api_url_or_params from (select active, address,	birthDate, communication, contact, extension, gender, generalPractitioner, id,	identifier,	managingOrganization,maritalStatus,meta,name,resourceType,	telecom, text,file_path,extraction_timestamp,api_url_or_params, row_number() over (partition by id order by meta.lastUpdated desc) as rn from workspace.raw.bronze_patient where id is not null) where rn=1''')

patient_incoming = (patient_deduplicate.withColumn("silver_processed_timestamp", to_timestamp(lit(load_date))).withColumn("is_current", lit(True)).withColumn("effective_start_date", to_timestamp(lit(load_date))).withColumn("effective_end_date", lit(None).cast("timestamp"))).createOrReplaceTempView('patient_incoming') 

spark.sql('''create table if not exists workspace.silver.silver_patient using delta as select * from patient_incoming where 1=0''')

spark.sql(f'''merge into workspace.silver.silver_patient s using patient_incoming i on s.id=i.id and s.is_current=true when matched and s.meta.lastUpdated<i.meta.lastUpdated then update set s.is_current=false,s.effective_end_date=cast('{load_date}' as timestamp)''')

patient_to_insert  = spark.sql('''select source.* from patient_incoming as source left join workspace.silver.silver_patient as target on source.id = target.id and target.is_current = TRUE
where target.id is null''')

patient_to_insert.write.format('delta').mode('append').saveAsTable('workspace.silver.silver_patient')

In [0]:
%sql
-- select count(id), lastUpdated from (select id,name,identifier,meta.lastUpdated as lastUpdated from workspace.raw.bronze_patient where meta.lastUpdated  = '2026-08-10T20:53:16.775-04:00') group by LastUpdated having count(id)>1

In [0]:
# Deduplicates patient records and maintains current patient versions in the Silver Delta table.

encounter_deduplicate = spark.sql('''select appointment,class,diagnosis,hospitalization,id,identifier,length,location,meta,participant,period,priority,reasonCode,resourceType,serviceProvider,status,subject,type,file_path,extraction_timestamp,api_url_or_params from (select appointment,class,diagnosis,hospitalization,id,identifier,length,location,meta,participant,period,priority,reasonCode,resourceType,serviceProvider,status,subject,type,file_path,extraction_timestamp,api_url_or_params, row_number() over (partition by id order by meta.lastUpdated desc) as rn  from workspace.raw.bronze_encounter where id is not null) where rn=1''')

encounter_incoming = (encounter_deduplicate.withColumn("silver_processed_timestamp", to_timestamp(lit(load_date))).withColumn("is_current", lit(True)).withColumn("effective_start_date", to_timestamp(lit(load_date))).withColumn("effective_end_date", lit(None).cast("timestamp"))).createOrReplaceTempView('encounter_incoming')

spark.sql('''create table if not exists workspace.silver.silver_encounter using delta as select * from encounter_incoming where 1=0''')

spark.sql(f'''merge into workspace.silver.silver_encounter s using encounter_incoming i on s.id=i.id and s.is_current=true when matched and s.meta.lastUpdated<i.meta.lastUpdated then update set s.is_current=false,s.effective_end_date=cast('{load_date}' as timestamp)''')

encounter_to_insert  = spark.sql('''select source.* from encounter_incoming as source left join workspace.silver.silver_encounter as target on source.id = target.id and target.is_current = TRUE
where target.id is null''')

encounter_to_insert.write.format('delta').mode('append').saveAsTable('workspace.silver.silver_encounter')

In [0]:
# Deduplicates obervation records and maintains current obervation versions in the Silver Delta table.

obervation_deduplicate = spark.sql('''select category,code,component,device,effectiveDateTime,effectivePeriod,encounter,id,identifier,interpretation,issued,meta,note,partOf,performer,referenceRange,resourceType,status,subject,text,valueBoolean,valueCodeableConcept from(select category,code,component,device,effectiveDateTime,effectivePeriod,encounter,id,identifier,interpretation,issued,meta,note,partOf,performer,referenceRange,resourceType,status,subject,text,valueBoolean,valueCodeableConcept, row_number() over(partition by id order by meta.lastUpdated) as rn from workspace.raw.bronze_observation where id is not null) where rn=1''')

observation_incoming = (obervation_deduplicate.withColumn("silver_processed_timestamp", to_timestamp(lit(load_date))).withColumn("is_current", lit(True)).withColumn("effective_start_date", to_timestamp(lit(load_date))).withColumn("effective_end_date", lit(None).cast("timestamp"))).createOrReplaceTempView('observation_incoming')

spark.sql('''create table if not exists workspace.silver.silver_observation using delta as select * from observation_incoming where 1=0''')

spark.sql(f'''merge into workspace.silver.silver_encounter s using observation_incoming i on s.id=i.id and s.is_current=true when matched and s.meta.lastUpdated<i.meta.lastUpdated then update set s.is_current=false,s.effective_end_date=cast('{load_date}' as timestamp)''')

observation_to_insert  = spark.sql('''select source.* from observation_incoming as source left join workspace.silver.silver_observation as target on source.id = target.id and target.is_current = TRUE
where target.id is null''')

observation_to_insert.write.format('delta').mode('overwrite').saveAsTable("workspace.silver.silver_observation")

In [0]:
# Deduplicates condition records and maintains current condition versions in the Silver Delta table.

condition_deduplicate = spark.sql('''select bodySite,category,clinicalStatus,code,encounter,evidence,id,identifier,meta,note,onsetDateTime,recordedDate,resourceType,severity,subject,verificationStatus,file_path,extraction_timestamp,api_url_or_params from (select bodySite,category,clinicalStatus,code,encounter,evidence,id,identifier,meta,note,onsetDateTime,recordedDate,resourceType,severity,subject,verificationStatus,file_path,extraction_timestamp,api_url_or_params,row_number() over (partition by id order by meta.lastUpdated desc) as rn from workspace.raw.bronze_condition where id is not null) where rn=1''')

condition_incoming = (condition_deduplicate.withColumn("silver_processed_timestamp", to_timestamp(lit(load_date))).withColumn("is_current", lit(True)).withColumn("effective_start_date", to_timestamp(lit(load_date))).withColumn("effective_end_date", lit(None).cast("timestamp"))).createOrReplaceTempView('condition_incoming')

spark.sql('''create table if not exists workspace.silver.silver_condition using delta as select * from condition_incoming where 1=0''')

spark.sql(f'''merge into workspace.silver.silver_condition s using condition_incoming i on s.id=i.id and s.is_current=true when matched and s.meta.lastUpdated<i.meta.lastUpdated then update set s.is_current=false,s.effective_end_date=cast('{load_date}' as timestamp)''')

condition_to_insert  = spark.sql('''select source.* from condition_incoming as source left join workspace.silver.silver_observation as target on source.id = target.id and target.is_current = TRUE
where target.id is null''')

condition_to_insert.write.format('delta').mode('overwrite').saveAsTable("workspace.silver.silver_condition")

In [0]:
dbutils.notebook.exit('success')